In [1]:
import pandas as pd
import os
import shutil
import pyarrow.parquet as pq
import duckdb as db

In [81]:
con = db.connect()

In [79]:
base = r"C:\Users\Nelio\Desktop\mds_Notas\Relatorio Notas\incrementais\ja_processados\notas"


In [80]:
incrementais = r"C:\Users\Nelio\Desktop\mds_Notas\Relatorio Notas\incrementais\ler"


In [ ]:
df = con.execute(f"""
    WITH base AS (
        SELECT 
            concat(Nota, CPF_CNPJ_Prestador, CPF_CNPJ_Tomador) AS ID,
            *
        FROM read_csv_auto('{base}/*.csv', delim=';', all_varchar=true)
    ),
    incremental AS (
        SELECT 
            concat(Nota, CPF_CNPJ_Prestador, CPF_CNPJ_Tomador) AS ID,
            *
        FROM read_csv_auto('{incrementais}/*.csv', delim=';', all_varchar=true)
    ),

    unificado as (
    SELECT * FROM incremental
    UNION
    SELECT * FROM base
    )

    SELECT * FROM unificado
    where id = '990252211031075200029308494546708'

    -- SELECT 
    --  base.ID as id_base,
    --  incremental.ID as id_inc
    --  FROM base
    --  LEFT JOIN incremental ON base.ID = incremental.ID
    --  WHERE incremental.ID IS not NULL

""").df()
df

,ID,Nota,CPF_CNPJ_Tomador,Nome_Tomador,Data_Emissao,Codigo_Item_Servico,Descricao_Servicos,Valor_da_Nota,Valor_Deducoes,Base_de_Calculo,Aliquota,Valor_ISS,Cancelada,CPF_CNPJ_Prestador,Tipo_Tributacao,Substituicao_Simples,Numero_Guia
0,990252211031075200029308494546708,99025221,08494546708,DELGADO DA MOTA EDUARDO,2025-12-07,09.01,ISS-ImpostosobreservicoDiariaSimples,337.44,0.00,337.44,5.00,16.87,NAO,10310752000293,TRIBUTADO NO MUNICIPIO,None,140538
1,990252211031075200029308494546708,99025221,08494546708,DELGADO DA MOTA EDUARDO,2025-12-07,09.01,ISS-ImpostosobreservicoDiariaSimples,337.44,0.00,337.44,5.00,16.87,NAO,10310752000293,TRIBUTADO NO MUNICIPIO,None,None


In [ ]:
69852 + 103483

# 990252211031075200029308494546708
# 990252231031075200029302553744000169
# 990252271031075200029331370836000101

173335

In [ ]:
def process_notas(df_base, inc_path):
    """Processa o incremental de Adição de Notas (Anti-Join)."""
    notas = ler_csv(inc_path)
    if notas is None or notas.empty:
        print("\n--- Adição de Notas Fiscais: Pulada (Arquivo vazio ou não encontrado). ---")
        return df_base

    print(f"\n--- INICIANDO PROCESSAMENTO: Adição de Notas Fiscais --- (Total: {len(notas)})")
    
    -- -- Anti-Join: Identifica o que é 'left_only' (novo)
    df_verificacao = notas.merge(
        df_base[CHAVE_UNICA], 
        on=CHAVE_UNICA,
        how='left',
        indicator=True
    )
    df_novas_linhas = df_verificacao.query('_merge == "left_only"').drop(columns=['_merge'])
    linhas_ignoradas = len(notas) - len(df_novas_linhas)
    
    if len(df_novas_linhas) > 0:
        df_base = pd.concat([df_base, df_novas_linhas], ignore_index=True)
        print(f"Linhas únicas adicionadas à base: {len(df_novas_linhas)}")
        print(f"Linhas ignoradas por duplicidade: {linhas_ignoradas}")
    else:
        print("Nenhuma linha nova encontrada para adição.")

    MOVER_ARQ(inc_path, 'notas')
    return df_base

In [ ]:
    -- 1. Processamento - Adição de Notas
    df_base = process_notas(df_base, INC_NOTA_PATH)